# Verificación de la capa Silver — Iceberg

Revisamos la tabla `nessie.nyc.yellow_taxi` tras la limpieza:
referencias Nessie, metadatos Iceberg, snapshots, particionado y calidad de datos.
También comparamos contra bronze para validar las reglas de limpieza.

In [1]:
from common.spark_factory import get_spark
from common import nessie_manager as nessie
from pyspark.sql.functions import (
    col, count, year as spark_year, month as spark_month,
    sum as spark_sum, round as spark_round, when,
)

spark = get_spark("verify-silver-iceberg")
spark.version

26/05/31 16:59:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.8'

## 1. Catálogo Nessie — Referencias y namespaces

In [2]:
print("Nessie references:\n")
nessie.print_references(spark)

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"
branches = refs_df.filter(refs_df[type_col].ilike("branch")).count()
tags     = refs_df.filter(refs_df[type_col].ilike("tag")).count()
print(f"\n  Branches: {branches}  |  Tags: {tags}")

Nessie references:

  * Branch bronze                                   6c92d3f09112
  * Branch main                                     e350bde574fc
  * Branch silver                                   18330668253e
    Tag    bronze-taxi_zone_lookup-2026             6c92d3f09112
    Tag    bronze-yellow_taxi-2023                  0ffe14008656
    Tag    bronze-yellow_taxi-2024                  fb478efd4d86
    Tag    bronze-yellow_taxi-2025                  ee10bf94a0f6
    Tag    silver-taxi_zone_lookup-2026             18330668253e
    Tag    silver-yellow_taxi-2023                  b422f845f9a3
    Tag    silver-yellow_taxi-2024                  64eb551c3c27
    Tag    silver-yellow_taxi-2025                  3f82d01ca96a

  Branches: 3  |  Tags: 8


In [3]:
# Cambiamos a la rama silver para el resto del notebook
nessie.set_spark_branch(spark, "silver")

spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)
spark.sql("SHOW TABLES IN nessie.nyc").show(truncate=False)

+---------+
|namespace|
+---------+
|nyc      |
+---------+

+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|nyc      |taxi_zone_lookup|false      |
|nyc      |yellow_taxi     |false      |
+---------+----------------+-----------+



## 2. Yellow Taxi — Propiedades y esquema

In [4]:
spark.sql("DESCRIBE EXTENDED nessie.nyc.yellow_taxi").show(250, truncate=False)

# DESCRIBE EXTENDED de Spark no surfacea las restricciones NOT NULL de
# Iceberg (la columna `comment` sale en blanco). Enumeramos las columnas
# NOT NULL y NULLABLE leyendo el schema del DataFrame, que es la fuente
# autoritativa de nullability.
fields = spark.table("nessie.nyc.yellow_taxi").schema.fields
not_null = [f.name for f in fields if not f.nullable]
nullable = [f.name for f in fields if f.nullable]
print(f"NOT NULL ({len(not_null)}):")
for n in not_null:
    print(f"  - {n}")
print()
print(f"NULLABLE ({len(nullable)}):")
for n in nullable:
    print(f"  - {n}")


+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                                                                     |comment|
+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
spark.sql("SHOW TBLPROPERTIES nessie.nyc.yellow_taxi").show(truncate=False)

+------------------------------------------+----------------------------------------------------------------+
|key                                       |value                                                           |
+------------------------------------------+----------------------------------------------------------------+
|current-snapshot-id                       |1019361955837351343                                             |
|format                                    |iceberg/parquet                                                 |
|format-version                            |2                                                               |
|gc.enabled                                |false                                                           |
|nessie.commit.id                          |18330668253e1be1520c24a44e86250417ddebacf8063a07cfa58c1e2439cea8|
|write.metadata.delete-after-commit.enabled|false                                                           |
|write.par

## 3. Snapshots e historial

In [6]:
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                       |summary                                                                                                                       

In [7]:
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-05-31 16:53:32.801|1973399907484904986|NULL               |true               |
|2026-05-31 16:54:08.384|7404693328164183664|1973399907484904986|true               |
|2026-05-31 16:54:43.549|5318224165263889170|7404693328164183664|true               |
|2026-05-31 16:55:17.1  |8768623358297634037|5318224165263889170|true               |
|2026-05-31 16:55:52.327|8358643046594428752|8768623358297634037|true               |
|2026-05-31 16:56:25.723|4727035024453091061|8358643046594428752|true               |
|2026-05-31 16:56:58.165|8153484920863139372|4727035024453091061|true               |
|2026-05-31 16:57:29.551|5743124165346433453|8153484920863139372|true               |
|2026-05-31 16:58:04.144|1019361955837351343|574312416

## 3b. Crecimiento de registros por tag

Cada tag `silver-yellow_taxi-{año}` captura el estado de `silver` tras procesar
todos los meses de ese año. Comparamos también con el tag equivalente en bronze.

In [8]:
import re

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"

silver_tags = [
    r["name"] for r in
    refs_df.filter(refs_df[type_col].ilike("tag"))
           .filter(refs_df["name"].rlike(r"^silver-yellow_taxi-\d{4}$"))
           .collect()
]
silver_tags.sort()

# Bronze ingiere ficheros completos del fuente que arrastran filas con pickup
# fuera del ámbito procesado por silver (años como 2001/2008/2014/2022 y meses
# fuera de Q1). Para que retention_pct compare manzanas con manzanas y los
# totales bronze cuadren con la sección 7, restringimos bronze al mismo
# ámbito que silver procesa: Q1 de los años válidos.
VALID_YEARS  = [2023, 2024, 2025]
VALID_MONTHS = [1, 2, 3]

tag_pattern = re.compile(r"silver-yellow_taxi-(\d{4})")
rows = []
prev_silver = 0
for tag in silver_tags:
    year = int(tag_pattern.match(tag).group(1))

    nessie.set_spark_branch(spark, tag)
    silver_total = spark.table("nessie.nyc.yellow_taxi").count()

    bronze_tag = f"bronze-yellow_taxi-{year}"
    try:
        nessie.set_spark_branch(spark, bronze_tag)
        bronze_total = (
            spark.table("nessie.nyc.yellow_taxi")
                 .filter(spark_year("tpep_pickup_datetime").isin(VALID_YEARS))
                 .filter(spark_month("tpep_pickup_datetime").isin(VALID_MONTHS))
                 .count()
        )
    except Exception:
        bronze_total = None

    retention = round(silver_total / bronze_total * 100, 2) if bronze_total else None
    rows.append((tag, year, bronze_total, silver_total, silver_total - prev_silver, retention))
    prev_silver = silver_total

nessie.set_spark_branch(spark, "silver")

growth_df = spark.createDataFrame(
    rows,
    ["tag", "year", "bronze_records", "silver_records", "records_added", "retention_pct"]
)
growth_df.show(truncate=False)


+-----------------------+----+--------------+--------------+-------------+-------------+
|tag                    |year|bronze_records|silver_records|records_added|retention_pct|
+-----------------------+----+--------------+--------------+-------------+-------------+
|silver-yellow_taxi-2023|2023|9384348       |8323188       |8323188      |88.69        |
|silver-yellow_taxi-2024|2024|18939105      |16312577      |7989389      |86.13        |
|silver-yellow_taxi-2025|2025|30137106      |24392616      |8080039      |80.94        |
+-----------------------+----+--------------+--------------+-------------+-------------+



## 4. Particionado y ficheros

In [9]:
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.partitions").show(truncate=False)

+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{648}    |0      |2565617     |1         |40727641                     |0                           |0                         |0                           |0                         |2026-05-31 16:55:17.1  |8768623358297634037     |
|{649}    |0      |2564751     |1         |40372015         

In [10]:
from pyspark.sql.functions import count, sum as spark_sum, round as spark_round

files_df = spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.files")
files_df.select(
    "content", "file_path", "file_format", "record_count", "file_size_in_bytes"
).show(100, truncate=80)

files_df.select(
    count("*").alias("total_files"),
    spark_round(spark_sum("file_size_in_bytes") / 1024 / 1024, 2).alias("total_size_mb"),
    spark_sum("record_count").alias("total_records"),
).show()

+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|content|                                                                       file_path|file_format|record_count|file_size_in_bytes|
+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|      0|s3a://warehouse/silver/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2025-0...|    PARQUET|     2906052|          46995037|
|      0|s3a://warehouse/silver/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2025-0...|    PARQUET|     2515289|          40386386|
|      0|s3a://warehouse/silver/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2025-0...|    PARQUET|     2658698|          42994173|
|      0|s3a://warehouse/silver/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2024-0...|    PARQUET|     2859021|          45483215|
|      0|s3a://warehouse/silver/nyc/yellow_taxi/data/tp

## 5. Calidad de datos — visión general

In [11]:
df = spark.table("nessie.nyc.yellow_taxi").cache()
print(f"Total records: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

Total records: 24,392,616
Columns: 20
root
 |-- VendorID: long (nullable = false)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = false)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = false)
 |-- passenger_count: double (nullable = false)
 |-- trip_distance: double (nullable = false)
 |-- RatecodeID: double (nullable = false)
 |-- store_and_fwd_flag: string (nullable = false)
 |-- PULocationID: long (nullable = false)
 |-- DOLocationID: long (nullable = false)
 |-- payment_type: long (nullable = false)
 |-- fare_amount: double (nullable = false)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = false)
 |-- tolls_amount: double (nullable = false)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = false)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [12]:
# Registros por año-mes (valida el particionado y la cobertura temporal)
df.groupBy(
    spark_year("tpep_pickup_datetime").alias("year"),
    spark_month("tpep_pickup_datetime").alias("month")
).count() \
 .orderBy("year", "month") \
 .show(50)

+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2023|    1|2728950|
|2023|    2|2585148|
|2023|    3|3009090|
|2024|    1|2565617|
|2024|    2|2564751|
|2024|    3|2859021|
|2025|    1|2658698|
|2025|    2|2515289|
|2025|    3|2906052|
+----+-----+-------+



In [13]:
# Nulos por columna. Cuatro comportamientos distintos:
#  - Columnas que clean() filtra (datetimes, IDs, importes con > 0 / >= 0,
#    passenger_count, RatecodeID, store_and_fwd_flag, ...): son NOT NULL en
#    silver, así que 0 es lo esperado.
#  - congestion_surcharge y airport_fee: NULLABLE pero 0 aquí. En el origen
#    sus NULL coinciden exactamente con el bloque de viajes que no informa
#    passenger_count / RatecodeID / store_and_fwd_flag; como clean() descarta
#    ese bloque (between(1,6) / isin), ningún NULL suyo sobrevive a silver.
#  - cbd_congestion_fee (introducida en 2025): NULL en todas las filas de
#    2023 y 2024 (anteriores a su introducción) y poblada en 2025 — la única
#    columna con NULL en silver. Su recuento cuadra con el total silver de
#    2023 + 2024.
#  - extra, mta_tax, improvement_surcharge: no filtrados por clean(), NULL
#    permitido aunque la fuente suela informarlos (0 aquí).
null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts.show(vertical=True)


-RECORD 0-------------------------
 VendorID              | 0        
 tpep_pickup_datetime  | 0        
 tpep_dropoff_datetime | 0        
 passenger_count       | 0        
 trip_distance         | 0        
 RatecodeID            | 0        
 store_and_fwd_flag    | 0        
 PULocationID          | 0        
 DOLocationID          | 0        
 payment_type          | 0        
 fare_amount           | 0        
 extra                 | 0        
 mta_tax               | 0        
 tip_amount            | 0        
 tolls_amount          | 0        
 improvement_surcharge | 0        
 total_amount          | 0        
 congestion_surcharge  | 0        
 airport_fee           | 0        
 cbd_congestion_fee    | 16312577 



In [14]:
# Estadísticas descriptivas
df.select(
    "trip_distance", "fare_amount", "tip_amount",
    "tolls_amount", "total_amount", "passenger_count"
).summary("count", "min", "max", "mean", "stddev").show()

26/05/31 17:00:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+------------------+------------------+-----------------+------------------+
|summary|     trip_distance|       fare_amount|        tip_amount|      tolls_amount|     total_amount|   passenger_count|
+-------+------------------+------------------+------------------+------------------+-----------------+------------------+
|  count|          24392616|          24392616|          24392616|          24392616|         24392616|          24392616|
|    min|               0.0|              0.01|               0.0|               0.0|             1.01|               1.0|
|    max|             69.93|             378.9|             161.0|            196.99|           453.05|               6.0|
|   mean|2.6855552430290315|16.360159121105777| 3.131807550285303|0.3285384031788008|24.73530273094258| 1.342429651661798|
| stddev|3.6544699584983893|14.813171977683394|3.2549837600724314| 1.718438779111486|18.36285076704834|0.8187723925612287|
+-------+-------

## 6. Validación de las reglas de limpieza

Cada check debe devolver **0 registros**. Si alguno devuelve filas, la regla de limpieza falló.

In [15]:
# Reglas de limpieza consolidadas en una sola pasada sobre df.
# Cada celda devolvía antes una cuenta independiente (1 escaneo por filtro);
# aquí se evalúan todas a la vez y df está cacheado.
# fare_amount / total_amount: silver exige "> 0" (un viaje real nunca cuesta 0),
# así que el check correcto es <= 0, no < 0. La regla de outliers IQR se verifica
# aparte en la sección 6b.
VALID_YEARS         = [2023, 2024, 2025]
VALID_VENDOR_IDS    = [1, 2, 6, 7]
VALID_RATE_CODES    = [1, 2, 3, 4, 5, 6, 99]
VALID_PAYMENT_TYPES = [0, 1, 2, 3, 4, 5, 6]

checks = {
    "null_pickup":            col("tpep_pickup_datetime").isNull(),
    "null_dropoff":           col("tpep_dropoff_datetime").isNull(),
    "pickup_ge_dropoff":      col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime"),
    "wrong_year":             ~spark_year("tpep_pickup_datetime").isin(VALID_YEARS),
    "neg_trip_distance":      col("trip_distance") < 0,
    "nonpos_fare_amount":     col("fare_amount")   <= 0,
    "nonpos_total_amount":    col("total_amount")  <= 0,
    "neg_tip_amount":         col("tip_amount")    < 0,
    "neg_tolls_amount":       col("tolls_amount")  < 0,
    "invalid_pu_location":    ~col("PULocationID").between(1, 265),
    "invalid_do_location":    ~col("DOLocationID").between(1, 265),
    "invalid_vendor":         ~col("VendorID").isin(VALID_VENDOR_IDS),
    "invalid_ratecode":       ~col("RatecodeID").isin(VALID_RATE_CODES),
    "invalid_payment_type":   ~col("payment_type").isin(VALID_PAYMENT_TYPES),
    "invalid_store_fwd_flag": ~col("store_and_fwd_flag").isin("Y", "N"),
    "invalid_passenger_cnt":  ~col("passenger_count").between(1, 6),
}

results = df.agg(*[
    spark_sum(when(cond, 1).otherwise(0)).alias(label)
    for label, cond in checks.items()
]).collect()[0].asDict()

width = max(len(k) for k in results)
for label, n in results.items():
    flag = "OK" if n == 0 else "FAIL"
    print(f"  [{flag}] {label.ljust(width)} : {n}")

  [OK] null_pickup            : 0
  [OK] null_dropoff           : 0
  [OK] pickup_ge_dropoff      : 0
  [OK] wrong_year             : 0
  [OK] neg_trip_distance      : 0
  [OK] nonpos_fare_amount     : 0
  [OK] nonpos_total_amount    : 0
  [OK] neg_tip_amount         : 0
  [OK] neg_tolls_amount       : 0
  [OK] invalid_pu_location    : 0
  [OK] invalid_do_location    : 0
  [OK] invalid_vendor         : 0
  [OK] invalid_ratecode       : 0
  [OK] invalid_payment_type   : 0
  [OK] invalid_store_fwd_flag : 0
  [OK] invalid_passenger_cnt  : 0


In [16]:
# Distribución de los campos categóricos
for field in ["VendorID", "RatecodeID", "payment_type", "store_and_fwd_flag", "passenger_count"]:
    print(f"--- {field} ---")
    df.groupBy(field).count().orderBy(field).show()

--- VendorID ---
+--------+--------+
|VendorID|   count|
+--------+--------+
|       1| 5937297|
|       2|18455319|
+--------+--------+

--- RatecodeID ---
+----------+--------+
|RatecodeID|   count|
+----------+--------+
|       1.0|23010570|
|       2.0|  834960|
|       3.0|   69868|
|       4.0|   51075|
|       5.0|  161705|
|       6.0|      32|
|      99.0|  264406|
+----------+--------+

--- payment_type ---
+------------+--------+
|payment_type|   count|
+------------+--------+
|           1|20449079|
|           2| 3595688|
|           3|  114332|
|           4|  233517|
+------------+--------+

--- store_and_fwd_flag ---
+------------------+--------+
|store_and_fwd_flag|   count|
+------------------+--------+
|                 N|24277482|
|                 Y|  115134|
+------------------+--------+

--- passenger_count ---
+---------------+--------+
|passenger_count|   count|
+---------------+--------+
|            1.0|19209796|
|            2.0| 3497920|
|            3.0|  

## 6b. Regla de outliers IQR

Tras el filtrado de validez, silver descarta por mes las filas fuera de las
vallas de Tukey `[Q1 − 3·IQR, Q3 + 3·IQR]` sobre `fare_amount`, `total_amount`,
`trip_distance` y la duración del viaje, calculadas **por `RatecodeID`** porque
cada código tarifario es una población distinta (metro estándar, tarifa plana
JFK/Newark, negociada, grupo, desconocida). Ver `clean_yellow_taxi.py`.

Verificación: para cada mes presente en silver se reconstruye la rodaja
bronze ya validada (los mismos filtros que `clean()`), se recalculan las
vallas `k = 3` por `RatecodeID` con el mismo `percentile_approx` / `accuracy`
que el pipeline, y se comprueba que **ninguna** fila de silver queda fuera de
la valla de **su propio** `RatecodeID`. El recuento esperado es 0.

In [17]:
# Verificación fiel de la regla IQR por RatecodeID: por cada mes de silver
# reconstruimos las vallas de Tukey k=3 por código tarifario desde la rodaja
# bronze validada y comprobamos que ninguna fila de silver queda fuera de la
# valla de su propio RatecodeID. clearCache() evita la colisión de
# InMemoryRelation entre refs bronze/silver (ver celda de la sección 7).
from pyspark.sql.functions import lit, percentile_approx, unix_timestamp

spark.catalog.clearCache()

IQR_K                   = 3.0
QUANTILE_RELATIVE_ERROR = 0.001
QUANTILE_ACCURACY       = int(1 / QUANTILE_RELATIVE_ERROR)
VALID_VENDOR_IDS    = [1, 2, 6, 7]
VALID_RATE_CODES    = [1, 2, 3, 4, 5, 6, 99]
VALID_PAYMENT_TYPES = [0, 1, 2, 3, 4, 5, 6]

# Duración del viaje (s) — base de la valla IQR de duración (no se almacena).
DURATION = (unix_timestamp("tpep_dropoff_datetime")
            - unix_timestamp("tpep_pickup_datetime"))
OUTLIER_COLS = ["fare_amount", "total_amount", "trip_distance", "_dur"]


def validity_filtered(d):
    """Etapa de validez de clean() — la rodaja sobre la que el pipeline
    calcula las vallas IQR por RatecodeID de ese mes."""
    return (
        d.filter(col("tpep_pickup_datetime").isNotNull())
         .filter(col("tpep_dropoff_datetime").isNotNull())
         .filter(col("tpep_pickup_datetime") < col("tpep_dropoff_datetime"))
         .dropDuplicates()
         .filter(col("trip_distance") >= 0)
         .filter(col("fare_amount") > 0)
         .filter(col("total_amount") > 0)
         .filter(col("tip_amount") >= 0)
         .filter(col("tolls_amount") >= 0)
         .filter(col("PULocationID").between(1, 265))
         .filter(col("DOLocationID").between(1, 265))
         .filter(col("VendorID").isin(VALID_VENDOR_IDS))
         .filter(col("RatecodeID").isin(VALID_RATE_CODES))
         .filter(col("payment_type").isin(VALID_PAYMENT_TYPES))
         .filter(col("store_and_fwd_flag").isin("Y", "N"))
         .filter(col("passenger_count").between(1, 6))
    )


def outside_fence_expr(fences):
    """Para cada columna se construye un CASE WHEN por RatecodeID con la
    valla calculada; la disyunción de los cuatro chequeos marca filas fuera de
    al menos una valla. Replica exactamente la lógica de remove_outliers()."""
    outside = lit(False)
    for c in OUTLIER_COLS:
        col_outside = lit(False)  # default: no hay valla -> no es violación
        for r in fences:
            q1, q3 = r[c]
            iqr = q3 - q1
            lo, hi = q1 - IQR_K * iqr, q3 + IQR_K * iqr
            col_outside = when(
                col("RatecodeID") == r["RatecodeID"],
                ~col(c).between(lo, hi),
            ).otherwise(col_outside)
        outside = outside | col_outside
    return outside


nessie.set_spark_branch(spark, "silver")
months = sorted(
    (r["y"], r["m"]) for r in
    spark.table("nessie.nyc.yellow_taxi")
         .select(spark_year("tpep_pickup_datetime").alias("y"),
                 spark_month("tpep_pickup_datetime").alias("m"))
         .distinct().collect()
)

violations_total = 0
print(f"{'mes':<10} {'fuera de valla IQR':>20}")
print("-" * 31)
for yr, mo in months:
    month_sel = ((spark_year("tpep_pickup_datetime") == yr)
                 & (spark_month("tpep_pickup_datetime") == mo))

    # Vallas por RatecodeID desde la rodaja bronze validada del mes.
    # collect() las materializa a Python: ningún cross-ref de Nessie las afecta.
    nessie.set_spark_branch(spark, "bronze")
    fences = (validity_filtered(spark.table("nessie.nyc.yellow_taxi").filter(month_sel))
              .withColumn("_dur", DURATION)
              .groupBy("RatecodeID").agg(
                  *[percentile_approx(c, [0.25, 0.75], QUANTILE_ACCURACY).alias(c)
                    for c in OUTLIER_COLS])
              .collect())

    # Silver del mes -> filas fuera de la valla de su propio RatecodeID.
    nessie.set_spark_branch(spark, "silver")
    n = (spark.table("nessie.nyc.yellow_taxi").filter(month_sel)
              .withColumn("_dur", DURATION)
              .filter(outside_fence_expr(fences))
              .count())
    violations_total += n
    print(f"{yr}-{mo:02d}     {n:>20,}")

flag = "OK" if violations_total == 0 else "FAIL"
print(f"\n  [{flag}] filas de silver fuera de su valla IQR por RatecodeID: {violations_total:,}")

mes          fuera de valla IQR
-------------------------------
2023-01                        0
2023-02                        0
2023-03                        0
2024-01                        0
2024-02                        0
2024-03                        0
2025-01                        0
2025-02                        0
2025-03                        0

  [OK] filas de silver fuera de su valla IQR por RatecodeID: 0


## 7. Comparativa Bronze vs Silver por mes

Muestra cuántas filas se eliminaron en la limpieza, mes a mes.

In [18]:
# Solo meses dentro del ámbito procesado por silver. Bronze trae filas con pickup en
# años/meses fuera de ámbito (2001, 2008, 2014, 2022, ...) que ensucian la tabla;
# las filtramos para que la comparativa muestre sólo lo que silver intenta procesar.
#
# Entre ramas: la sesión Spark está ligada a una sola ref Nessie a la vez, así que
# materializamos (collect) las cuentas de cada rama antes de cambiar de ref.
#
# clearCache() es imprescindible: la celda 17 cacheó silver.yellow_taxi. El
# CacheManager reutiliza ese InMemoryRelation por igualdad de plan lógico y, como
# las SparkTable de bronze y silver son iguales (mismo nombre Iceberg, snapshotId
# nulo) y la ref Nessie es estado de sesión —no entra en el plan—, spark.table(...)
# sobre bronze recibiría el relation cacheado de silver: la comparativa saldría
# 100% en todos los meses. Limpiar la caché lo evita.
spark.catalog.clearCache()

VALID_YEARS  = [2023, 2024, 2025]
VALID_MONTHS = [1, 2, 3]

def monthly_counts(branch: str) -> dict:
    nessie.set_spark_branch(spark, branch)
    rows = (
        spark.table("nessie.nyc.yellow_taxi")
        .filter(spark_year("tpep_pickup_datetime").isin(VALID_YEARS))
        .filter(spark_month("tpep_pickup_datetime").isin(VALID_MONTHS))
        .groupBy(
            spark_year("tpep_pickup_datetime").alias("year"),
            spark_month("tpep_pickup_datetime").alias("month")
        )
        .agg(count("*").alias("n"))
        .collect()
    )
    return {(r["year"], r["month"]): r["n"] for r in rows}

bronze_by_month = monthly_counts("bronze")
silver_by_month = monthly_counts("silver")
nessie.set_spark_branch(spark, "silver")

rows = []
for ym in sorted(bronze_by_month.keys() | silver_by_month.keys()):
    b = bronze_by_month.get(ym, 0)
    s = silver_by_month.get(ym, 0)
    retention = round(s / b * 100, 2) if b else None
    rows.append((ym[0], ym[1], b, s, b - s, retention))

comparison = spark.createDataFrame(
    rows, ["year", "month", "bronze_count", "silver_count", "dropped", "retention_pct"]
).orderBy("year", "month").cache()

comparison.show(50)

+----+-----+------------+------------+-------+-------------+
|year|month|bronze_count|silver_count|dropped|retention_pct|
+----+-----+------------+------------+-------+-------------+
|2023|    1|     3066726|     2728950| 337776|        88.99|
|2023|    2|     2914003|     2585148| 328855|        88.71|
|2023|    3|     3403619|     3009090| 394529|        88.41|
|2024|    1|     2964617|     2565617| 399000|        86.54|
|2024|    2|     3007533|     2564751| 442782|        85.28|
|2024|    3|     3582607|     2859021| 723586|         79.8|
|2025|    1|     3475234|     2658698| 816536|         76.5|
|2025|    2|     3577542|     2515289|1062253|        70.31|
|2025|    3|     4145225|     2906052|1239173|        70.11|
+----+-----+------------+------------+-------+-------------+



In [19]:
# Totales agregados (reutiliza la comparativa cacheada en la celda anterior).
nessie.set_spark_branch(spark, "silver")
comparison.select(
    spark_sum("bronze_count").alias("total_bronze"),
    spark_sum("silver_count").alias("total_silver"),
    spark_sum("dropped").alias("total_dropped"),
    spark_round(
        spark_sum("silver_count") / spark_sum("bronze_count") * 100, 2
    ).alias("overall_retention_pct")
).show()

comparison.unpersist()
_ = df.unpersist()

+------------+------------+-------------+---------------------+
|total_bronze|total_silver|total_dropped|overall_retention_pct|
+------------+------------+-------------+---------------------+
|    30137106|    24392616|      5744490|                80.94|
+------------+------------+-------------+---------------------+



## 7b. Meses en cuarentena

Meses cuyo snapshot WAP falló la auditoría silver y se congeló como tag `review/silver-*`
en lugar de fusionarse. En la comparativa anterior se reconocen porque tienen
`silver_count = 0`, `dropped = bronze_count` y `retention_pct = 0`: bronze los ingirió
pero ninguno llegó a silver.

In [20]:
# Meses en cuarentena: cada tag review/silver-* es un mes que falló la auditoría
# silver. Su snapshot quedó congelado para inspección y NO se fusionó a silver
# -> explica los meses con silver_count = 0 en la comparativa de arriba.
refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"

review_tags = sorted(
    r["name"] for r in
    refs_df.filter(refs_df[type_col].ilike("tag"))
           .filter(refs_df["name"].rlike(r"^review/silver-"))
           .collect()
)

if review_tags:
    print(f"Meses en cuarentena ({len(review_tags)}):")
    for t in review_tags:
        print(f"  - {t}")
else:
    print("Sin meses en cuarentena: todos los meses superaron la auditoría silver.")

Sin meses en cuarentena: todos los meses superaron la auditoría silver.


## 7c. Auditoría de cuarentena — regla incumplida

Para cada mes congelado en `review/silver-*` reconstruimos el veredicto de la
auditoría silver (`drop_ratio` frente a `MAX_DROP_RATIO = 30 %`, definido en
`src/pipelines/silver/clean_yellow_taxi.py`) y desglosamos cuántas filas
elimina **cada regla de limpieza por separado** (validez y outliers IQR por
`RatecodeID`), evaluada de forma aislada sobre el bronze del mes.

La regla con más filas descartadas es la causa dominante del rechazo. Los
recuentos por regla pueden solaparse — una misma fila puede incumplir varias —,
por lo que su suma supera el total real de filas que elimina `clean()`. Las
vallas IQR se recalculan con el mismo `percentile_approx` / `accuracy` que el
pipeline.

In [21]:
# Reglas y umbral viven en src/pipelines/silver/clean_yellow_taxi.py.
# clearCache() evita la colisión de InMemoryRelation entre refs (ver celda 25).
import re
from functools import reduce
from pyspark.sql.functions import lit, percentile_approx, unix_timestamp

spark.catalog.clearCache()

MAX_DROP_RATIO          = 0.3
IQR_K                   = 3.0
QUANTILE_RELATIVE_ERROR = 0.001
QUANTILE_ACCURACY       = int(1 / QUANTILE_RELATIVE_ERROR)
VALID_VENDOR_IDS    = [1, 2, 6, 7]
VALID_RATE_CODES    = [1, 2, 3, 4, 5, 6, 99]
VALID_PAYMENT_TYPES = [0, 1, 2, 3, 4, 5, 6]

# Meses en cuarentena: derivados de los tags review/silver-yellow_taxi-* (celda 7b),
# no hardcodeados — así la auditoría sigue siendo válida tras reejecutar el pipeline.
refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"
tag_re = re.compile(r"^review/silver-yellow_taxi-(\d{4})-(\d{2})$")
QUARANTINED = sorted(
    (int(m.group(1)), int(m.group(2)))
    for m in (tag_re.match(r["name"]) for r in
              refs_df.filter(refs_df[type_col].ilike("tag")).collect())
    if m
)

# Etapa de validez de clean(): la fila sobrevive sólo si todas son verdaderas.
# when(keep, 0).otherwise(1) cuenta como descartada la fila que incumple y la
# que tiene NULL (la comparación con NULL es falsa -> el filtro la elimina).
VALIDITY_RULES = {
    "pickup no nulo":           col("tpep_pickup_datetime").isNotNull(),
    "dropoff no nulo":          col("tpep_dropoff_datetime").isNotNull(),
    "pickup < dropoff":         col("tpep_pickup_datetime") < col("tpep_dropoff_datetime"),
    "trip_distance >= 0":       col("trip_distance") >= 0,
    "fare_amount > 0":          col("fare_amount") > 0,
    "total_amount > 0":         col("total_amount") > 0,
    "tip_amount >= 0":          col("tip_amount") >= 0,
    "tolls_amount >= 0":        col("tolls_amount") >= 0,
    "PULocationID en [1,265]":  col("PULocationID").between(1, 265),
    "DOLocationID en [1,265]":  col("DOLocationID").between(1, 265),
    "VendorID válido":          col("VendorID").isin(VALID_VENDOR_IDS),
    "RatecodeID válido":        col("RatecodeID").isin(VALID_RATE_CODES),
    "payment_type válido":      col("payment_type").isin(VALID_PAYMENT_TYPES),
    "store_and_fwd_flag (Y/N)": col("store_and_fwd_flag").isin("Y", "N"),
    "passenger_count en [1,6]": col("passenger_count").between(1, 6),
}

# Duración del viaje (s) — base de la valla IQR de duración (no se almacena).
DURATION = (unix_timestamp("tpep_dropoff_datetime")
            - unix_timestamp("tpep_pickup_datetime"))
IQR_OUTLIER_COLS = ["fare_amount", "total_amount", "trip_distance", "_dur"]


def validity_filtered(d):
    keep = reduce(lambda a, b: a & b, VALIDITY_RULES.values())
    return d.filter(keep).dropDuplicates()


def iqr_violation_expr(c, fences):
    """1 si el valor de c queda fuera de la valla de su RatecodeID, 0 en caso
    contrario (también 0 si no hay valla: código sin match)."""
    expr = lit(0)
    for r in fences:
        q1, q3 = r[c]
        iqr = q3 - q1
        lo, hi = q1 - IQR_K * iqr, q3 + IQR_K * iqr
        expr = when(
            col("RatecodeID") == r["RatecodeID"],
            when(col(c).between(lo, hi), 0).otherwise(1),
        ).otherwise(expr)
    return expr


if not QUARANTINED:
    print("Sin meses en cuarentena: todos los meses superaron la auditoría silver.")

for yr, mo in QUARANTINED:
    tag = f"review/silver-yellow_taxi-{yr}-{mo:02d}"
    month_sel = ((spark_year("tpep_pickup_datetime") == yr)
                 & (spark_month("tpep_pickup_datetime") == mo))

    # Bronze del mes: mismo filtro year+month que aplica el pipeline.
    nessie.set_spark_branch(spark, "bronze")
    bm = spark.table("nessie.nyc.yellow_taxi").filter(month_sel).cache()
    bronze_count = bm.count()

    # Vallas IQR k=3 por RatecodeID, calculadas sobre la rodaja ya validada
    # del mes y materializadas a Python con collect() (igual que el pipeline).
    fences = (validity_filtered(bm).withColumn("_dur", DURATION)
              .groupBy("RatecodeID").agg(
                  *[percentile_approx(c, [0.25, 0.75], QUANTILE_ACCURACY).alias(c)
                    for c in IQR_OUTLIER_COLS])
              .collect())

    # Filas descartadas por cada regla, evaluada de forma aislada sobre bronze.
    bm_dur = bm.withColumn("_dur", DURATION)
    rule_exprs = {
        **{name: when(keep, 0).otherwise(1) for name, keep in VALIDITY_RULES.items()},
        "IQR fare_amount":   iqr_violation_expr("fare_amount", fences),
        "IQR total_amount":  iqr_violation_expr("total_amount", fences),
        "IQR trip_distance": iqr_violation_expr("trip_distance", fences),
        "IQR duración":      iqr_violation_expr("_dur", fences),
    }
    per_rule = bm_dur.agg(*[
        spark_sum(expr).alias(name) for name, expr in rule_exprs.items()
    ]).collect()[0].asDict()
    # dropDuplicates() no es una condición de columna: se mide aparte.
    per_rule["filas duplicadas"] = bronze_count - bm.dropDuplicates().count()
    bm.unpersist()

    # Veredicto real: el tag review/ congela el snapshot WAP ya limpiado, así que
    # su recuento del mes es el silver_count que evaluó validate().
    nessie.set_spark_branch(spark, tag)
    silver_count = spark.table("nessie.nyc.yellow_taxi").filter(month_sel).count()

    drop_ratio = (bronze_count - silver_count) / bronze_count
    dominant = max(per_rule, key=per_rule.get)

    print(f"\n=== {yr}-{mo:02d}  ({tag}) ===")
    print(f"  Bronze: {bronze_count:,}   Silver tras clean(): {silver_count:,}")
    print(f"  Drop ratio: {drop_ratio:.1%}   Umbral: {MAX_DROP_RATIO:.0%}   "
          f"-> {'RECHAZADO' if drop_ratio > MAX_DROP_RATIO else 'OK'}")
    print("  Filas descartadas por regla (aislada; los solapamientos hacen que\n"
          "  la suma supere el total real de clean()):")
    for name, n in sorted(per_rule.items(), key=lambda kv: -kv[1]):
        marca = "  <-- regla dominante" if name == dominant else ""
        print(f"    {name:<26} {n:>10,}  ({n / bronze_count * 100:5.2f} %){marca}")

_ = nessie.set_spark_branch(spark, "silver")

Sin meses en cuarentena: todos los meses superaron la auditoría silver.


### Conclusión

El desglose anterior atribuye cada rechazo a su **regla dominante**: la que más
filas elimina de forma aislada. Con el pipeline actual las causas habituales de
un `drop_ratio` alto son dos: (a) el bloque de viajes de 2025 que no informa
`passenger_count`, `RatecodeID` ni `store_and_fwd_flag` —tres reglas que golpean
las mismas filas, un cambio en el formato de reporte de la fuente—, y (b) la
regla de outliers IQR, que recorta las colas de `fare_amount`, `total_amount`,
`trip_distance` y duración.

Las cifras concretas dependen de la última corrida del pipeline; reejecuta estas
celdas tras cada ejecución de `clean_yellow_taxi.py` para reinterpretar la
cuarentena vigente. Si no hay tags `review/silver-*`, ningún mes fue rechazado.

## 8. Muestra de datos

In [22]:
spark.table("nessie.nyc.yellow_taxi").show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2025-01-01 00:42:40 |2025-01-01 00:55:38  |3.0            |1.99         |1.0       |N                 |236         |151 

## 9. Taxi zone lookup — esquema y calidad


In [23]:
zones = spark.table("nessie.nyc.taxi_zone_lookup")
print(f"Records: {zones.count()}")
zones.printSchema()
spark.sql("DESCRIBE EXTENDED nessie.nyc.taxi_zone_lookup").show(50, truncate=False)


Records: 265
root
 |-- location_id: integer (nullable = false)
 |-- borough: string (nullable = false)
 |-- zone: string (nullable = false)
 |-- service_zone: string (nullable = true)

+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                                                                     |comment|
+-----------------------------+--------------------------------------------------------------------------------------------------

In [24]:
# Nulos en campos críticos y rango del location_id
print("Null location_id:", zones.filter(col("location_id").isNull()).count())
print("Null borough   :", zones.filter(col("borough").isNull() | (col("borough") == "")).count())
print("Null zone      :", zones.filter(col("zone").isNull() | (col("zone") == "")).count())
print("Out-of-range location_id:",
      zones.filter(~col("location_id").between(1, 265)).count())

# Duplicados por location_id (clave lógica)
dupes = (zones.groupBy("location_id").count()
         .filter(col("count") > 1))
n_dupes = dupes.count()
print("Duplicate location_id:", n_dupes)
if n_dupes > 0:
    dupes.show()

Null location_id: 0
Null borough   : 0
Null zone      : 0
Out-of-range location_id: 0
Duplicate location_id: 0


In [25]:
# Distribución por borough y service_zone
print("--- borough ---")
zones.groupBy("borough").count().orderBy("borough").show(truncate=False)
print("--- service_zone ---")
zones.groupBy("service_zone").count().orderBy("service_zone").show(truncate=False)


--- borough ---
+-------------+-----+
|borough      |count|
+-------------+-----+
|Bronx        |43   |
|Brooklyn     |61   |
|EWR          |1    |
|Manhattan    |69   |
|N/A          |1    |
|Queens       |69   |
|Staten Island|20   |
|Unknown      |1    |
+-------------+-----+

--- service_zone ---
+------------+-----+
|service_zone|count|
+------------+-----+
|Airports    |2    |
|Boro Zone   |205  |
|EWR         |1    |
|N/A         |2    |
|Yellow Zone |55   |
+------------+-----+



In [26]:
# Comparativa Bronze vs Silver (la tabla de zonas es pequeña, ~265 filas: comparamos conjuntos de IDs)
nessie.set_spark_branch(spark, "bronze")
bronze_ids = {row["LocationID"] for row in
              spark.table("nessie.nyc.taxi_zone_lookup").select("LocationID").collect()}
nessie.set_spark_branch(spark, "silver")
silver_ids = {row["location_id"] for row in zones.select("location_id").collect()}

b, s = len(bronze_ids), len(silver_ids)
if b:
    print(f"Bronze: {b}  Silver: {s}  Dropped: {b - s}  Retention: {s / b:.1%}")
else:
    print("Bronze empty")

missing = sorted(bronze_ids - silver_ids)
print(f"IDs in bronze but not in silver: {len(missing)}")
if missing:
    print("  " + ", ".join(str(i) for i in missing))


Bronze: 265  Silver: 265  Dropped: 0  Retention: 100.0%
IDs in bronze but not in silver: 0


In [27]:
# Integridad referencial: ¿todas las PULocationID/DOLocationID de silver.yellow_taxi
# existen en silver.taxi_zone_lookup?
trips = spark.table("nessie.nyc.yellow_taxi")

pu_missing = (trips.select(col("PULocationID").alias("location_id")).distinct()
              .join(zones.select("location_id"), "location_id", "left_anti"))
do_missing = (trips.select(col("DOLocationID").alias("location_id")).distinct()
              .join(zones.select("location_id"), "location_id", "left_anti"))

print("PULocationID not found in zone lookup:", pu_missing.count())
pu_missing.show(truncate=False)
print("DOLocationID not found in zone lookup:", do_missing.count())
do_missing.show(truncate=False)


PULocationID not found in zone lookup: 0
+-----------+
|location_id|
+-----------+
+-----------+

DOLocationID not found in zone lookup: 0
+-----------+
|location_id|
+-----------+
+-----------+



In [28]:
# Muestra
zones.orderBy("location_id").show(10, truncate=False)


+-----------+-------------+-----------------------+------------+
|location_id|borough      |zone                   |service_zone|
+-----------+-------------+-----------------------+------------+
|1          |EWR          |Newark Airport         |EWR         |
|2          |Queens       |Jamaica Bay            |Boro Zone   |
|3          |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4          |Manhattan    |Alphabet City          |Yellow Zone |
|5          |Staten Island|Arden Heights          |Boro Zone   |
|6          |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7          |Queens       |Astoria                |Boro Zone   |
|8          |Queens       |Astoria Park           |Boro Zone   |
|9          |Queens       |Auburndale             |Boro Zone   |
|10         |Queens       |Baisley Park           |Boro Zone   |
+-----------+-------------+-----------------------+------------+
only showing top 10 rows

